# Aviation Delay Prediction
**Lufthansa Systems — Data Science Challenge**

---

## 0. Executive Summary

This notebook documents an end-to-end analysis of a synthetically generated flight dataset with the goal of predicting whether a flight will be delayed by more than 15 minutes.

**Key findings:**
- The dataset contains **severe class imbalance** (96% no-delay / 4% delayed) — making accuracy a misleading metric.
- **5 columns were identified as data leakage** and removed before any modelling.
- To handle the severe imbalance, a **dynamic threshold sweep** was implemented, revealing the true predictive power of the ensemble models.
- The **Logistic Regression** achieves the best F1 (0.20) and a solid ROC-AUC of 0.7480, capturing 75% of all real delays.
- The most valuable outcome is the **analytical process**, identifying what we can and cannot trust in this data.

## 1. Setup

In [16]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.config import TARGET, DATE_COL, LEAKAGE_COLS, NUMERIC_COLS, CATEGORICAL_COLS
from src.data_loader import load_raw, report_leakage, drop_leakage, report_missing, time_split
from src.eda import (
    plot_target_distribution, plot_missing_values, plot_leakage_correlations,
    plot_numeric_distributions, plot_delays_over_time, plot_correlation_heatmap
)
from src.preprocessing import full_preprocess
from src.models import train_all, MODELS
from src.evaluation import (
    evaluate_all, save_results, print_summary_table,
    plot_model_comparison, plot_roc_curves, plot_confusion_matrices, plot_feature_importance
)

print('Setup complete.')

Setup complete.


## 2. Load & First Look

In [17]:
df = load_raw()
df.head()

[load_raw] Loaded 6,000 rows × 28 columns.


,flight_datetime,origin,destination,scheduled_departure_hour,route_distance,aircraft_type,aircraft_age,turnaround_minutes,passenger_load_factor,visibility,...,previous_leg_delay,maintenance_events_last_30d,crew_status_new_FINAL,sched_buffer_mins_latest,ops_delay_prediction_v2,actual_gate_out_time_diff,maintenance_closed_after_pushback,final_delay_reason,actual_delay_minutes,delay_over_15m
0,2024-01-01 00:00:00,CDG,ZRH,0,1208.039459,E190,12.331889,65.319296,0.895540,7.993844,...,0.000000,3.0,0,65.319296,13.148094,5.189946,0,NONE,2.373901,0
1,2024-01-01 00:30:00,VIE,MAD,0,1324.032304,E190,11.125741,49.239086,0.947061,10.471546,...,11.884488,NaN,0,49.239086,3.674381,3.509332,0,NONE,4.049822,0
2,2024-01-01 01:00:00,AMS,MUC,1,1198.270210,B737,12.015662,58.252149,0.932430,6.697409,...,22.047020,1.0,0,58.252149,14.527474,-3.781593,0,NONE,0.225041,0
3,2024-01-01 01:30:00,BUD,ZRH,1,1211.251089,E190,10.005903,42.246717,0.751699,11.642092,...,0.000000,1.0,0,42.246717,-1.375745,1.657266,0,NONE,8.432854,0
4,2024-01-01 02:00:00,CDG,MUC,2,1039.433763,E190,5.218236,73.213001,0.933868,9.213155,...,14.196942,NaN,0,73.213001,11.240716,-1.518433,0,NONE,1.237742,0


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 28 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   flight_datetime                    6000 non-null   datetime64[ns]
 1   origin                             6000 non-null   object        
 2   destination                        6000 non-null   object        
 3   scheduled_departure_hour           6000 non-null   int64         
 4   route_distance                     6000 non-null   float64       
 5   aircraft_type                      6000 non-null   object        
 6   aircraft_age                       6000 non-null   float64       
 7   turnaround_minutes                 6000 non-null   float64       
 8   passenger_load_factor              6000 non-null   float64       
 9   visibility                         5536 non-null   float64       
 10  wind_speed                         6

## 3. Data Leakage Analysis

**Before any EDA or modelling**, we must identify columns that would not be available at prediction time (i.e. when the flight has not yet departed or landed).

Including these would give the model "future information", producing artificially high scores that collapse to zero in production.


In [19]:
report_leakage(df)


=== Leakage diagnostic ===
actual_delay_minutes (num) → corr with target: +0.7471
actual_gate_out_time_diff (num) → corr with target: +0.7101
maintenance_closed_after_pushback (num) → corr with target: +0.3156
final_delay_reason (cat) → delay rate per category: {'ATC': 1.0, 'CREW': 1.0, 'MX': 1.0, 'NONE': 0.0, 'WX': 1.0}
sched_buffer_mins_latest (num) → corr with target: -0.0915



In [20]:
plot_leakage_correlations(df)

**Leakage columns identified and their justification:**

| Column | Correlation | Reason for exclusion |
|---|---|---|
| `actual_delay_minutes` | +0.747 | Target is directly derived from this |
| `actual_gate_out_time_diff` | +0.710 | Only measurable after gate-out |
| `maintenance_closed_after_pushback` | +0.316 | Post-pushback event |
| `final_delay_reason` | — | Categorical; assigned after delay occurs. NONE = no delay → perfect leakage |
| `sched_buffer_mins_latest` | -0.092 | 100% identical to `turnaround_minutes` —> redundant |

Note: `ops_delay_prediction_v2` is **kept**, despite its suspicious name, its correlation with the target is only 0.12, suggesting it is a genuine pre-flight estimate, not a post-hoc label.


In [21]:
df = drop_leakage(df)
print(f'Remaining columns: {df.shape[1]}')

[drop_leakage] Dropped 5 leakage columns: ['actual_delay_minutes', 'actual_gate_out_time_diff', 'maintenance_closed_after_pushback', 'final_delay_reason', 'sched_buffer_mins_latest']
Remaining columns: 23


## 4. Exploratory Data Analysis (EDA)

### 4.1 Target Distribution

In [22]:
plot_target_distribution(df)

> **Important:** Only 4% of flights are delayed >15 minutes. A naive model that always predicts "no delay" would achieve **96% accuracy**, completely useless in practice. This is why we use **F1-score** and **ROC-AUC** as primary metrics.

### 4.2 Missing Values

In [23]:
report_missing(df)
plot_missing_values(df)


=== Missing values ===
maintenance_events_last_30d: 722 (12.0%)
visibility: 464 (7.7%)



Two columns have missing values:
- `maintenance_events_last_30d`: 12.0% missing, likely not recorded for newer aircraft or short-haul routes
- `visibility`: 7.7% missing, possibly sensor gaps or unreported conditions

Both will be imputed with the **median** (fitted on training data only).

### 4.3 Temporal Patterns

In [24]:
plot_delays_over_time(df)

### 4.4 Feature Distributions by target

In [25]:
plot_numeric_distributions(df)

### 4.5 Correlation Heatmap (clean features only)

In [26]:
plot_correlation_heatmap(df)

## 5. Validation Strategy

Since `flight_datetime` is present, we apply a **time-based split** rather than random splitting.

**Why?** A random split would allow the model to train on flights from May while predicting March, impossible in production. We simulate deployment by always predicting the future from the past.

```
Timeline:  Jan ──── Feb ──── Mar ──── Apr ──── May
            [    TRAIN (4,368)     ] [VAL]   [TEST]
```


In [27]:
train, val, test = time_split(df)

[time_split] Train: 4,368 | Val: 1,440 | Test: 192
             Train target rate: 3.89%
             Val   target rate: 4.17%
             Test  target rate: 4.17%


## 6. Preprocessing

In [28]:
(
    X_train, X_val, X_test,
    X_train_sc, X_val_sc, X_test_sc,
    y_train, y_val, y_test,
) = full_preprocess(train, val, test)

print(f'X_train shape: {X_train.shape}')
print(f'Positive class rate — train: {y_train.mean():.2%} | val: {y_val.mean():.2%} | test: {y_test.mean():.2%}')

[impute] Numeric imputer (strategy=median) fitted on 4,368 rows.
[encode] One-hot encoded 3 columns → 25 dummies created.
[full_preprocess] Final feature matrix: 48 features.
X_train shape: (4368, 48)
Positive class rate — train: 3.89% | val: 4.17% | test: 4.17%


**Preprocessing steps applied:**
1. **Time features** extracted from `flight_datetime` (month, day_of_week, cyclical hour encoding)
2. **Median imputation** for `visibility` and `maintenance_events_last_30d`, fitted on train only
3. **One-hot encoding** for `origin`, `destination`, `aircraft_type`
4. **StandardScaler** for Logistic Regression, fitted on train only

> All transformations are fitted exclusively on training data to prevent subtle data leakage.

## 7. Modelling

We train 6 models across two tiers:

| Tier | Model | Rationale |
|---|---|---|
| Baseline | **DummyClassifier** | True floor, predicts majority class only |
| Baseline | **Logistic Regression** | Linear, interpretable, scaled inputs |
| Advanced | **Decision Tree** | Non-linear rules, human-readable |
| Advanced | **Random Forest** | Robust ensemble, reliable feature importance |
| Advanced | **XGBoost** | State-of-the-art on tabular data |
| Advanced | **LightGBM** | Handles missing values natively, fast |

**Class imbalance handling:** All models use either `class_weight='balanced'`, `scale_pos_weight` (XGBoost) or `is_unbalance=True` (LightGBM) to compensate for the 96/4 split.

In [29]:
fitted_models = train_all(X_train, X_val, X_train_sc, X_val_sc, y_train, y_val)

  Training: Dummy (majority)...
  Training: Logistic Regression...
  Training: Decision Tree...
  Training: Random Forest...
  Training: XGBoost...
  Training: LightGBM...

[train_all] Trained 6 models.


## 8. Results & Evaluation

In [30]:
df_results, preds = evaluate_all(
    fitted_models,
    X_val, X_val_sc, y_val,
    X_test, X_test_sc, y_test,
)
print_summary_table(df_results)
save_results(df_results)


--- threshold search on validation set ---
[Dummy (majority)] best threshold on val set: 0.5 (val F1=0.0000)
[Logistic Regression] best threshold on val set: 0.35 (val F1=0.1605)
[Decision Tree] best threshold on val set: 0.25 (val F1=0.1182)
[Random Forest] best threshold on val set: 0.4 (val F1=0.1575)
[XGBoost] best threshold on val set: 0.2 (val F1=0.1667)
[LightGBM] best threshold on val set: 0.15 (val F1=0.1457)

MODEL COMPARISON — TEST SET
                     accuracy  precision  recall      f1  roc_auc  threshold
model                                                                       
Dummy (majority)       0.9583     0.0000   0.000  0.0000   0.5000       0.50
Logistic Regression    0.7500     0.1154   0.750  0.2000   0.7480       0.35
Decision Tree          0.6198     0.0667   0.625  0.1205   0.5652       0.25
Random Forest          0.8958     0.0000   0.000  0.0000   0.7099       0.40
XGBoost                0.8750     0.0556   0.125  0.0769   0.6542       0.20
LightGBM 

In [31]:
plot_model_comparison(df_results)

[plot] Saved: e:\lufthansa-delay-prediction\outputs\figures\model_comparison.png


In [32]:
plot_roc_curves(fitted_models, preds, y_test)

[plot] Saved: e:\lufthansa-delay-prediction\outputs\figures\roc_curves.png


In [33]:
plot_confusion_matrices(preds, y_test)

[plot] Saved: e:\lufthansa-delay-prediction\outputs\figures\confusion_matrices.png


In [34]:
plot_feature_importance(fitted_models, list(X_train.columns))

[plot] Saved: e:\lufthansa-delay-prediction\outputs\figures\feature_importances.png


### Results Interpretation

Because of the extreme class imbalance (96/4), the default 0.5 decision
threshold renders most models overly conservative. To address this, a
**Threshold Sweep** was built into the pipeline: the optimal cut-off is
found on the **validation set** and then applied to the test set without
further adjustment, simulating a realistic production scenario where the
decision boundary is fixed before seeing new data.

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | Threshold (val) |
|---|---|---|---|---|---|---|
| Dummy (majority) | 0.958 | 0.000 | 0.000 | 0.000 | 0.500 | 0.50 |
| **Logistic Regression** | **0.750** | **0.115** | **0.750** | **0.200** | **0.748** | **0.35** |
| Decision Tree | 0.620 | 0.067 | 0.625 | 0.121 | 0.565 | 0.25 |
| Random Forest | 0.896 | 0.000 | 0.000 | 0.000 | 0.710 | 0.40 |
| XGBoost | 0.875 | 0.056 | 0.125 | 0.077 | 0.654 | 0.20 |
| LightGBM | 0.802 | 0.031 | 0.125 | 0.050 | 0.673 | 0.15 |

**Logistic Regression wins** with the highest F1 (0.200) and ROC-AUC
(0.748). Its `class_weight='balanced'` parameter shifts predicted
probabilities toward the minority class, and a val-set threshold of 0.35
is enough to activate positive predictions on the test set.

**Random Forest: a cautionary result.** On the validation set, the optimal
threshold was 0.40 (val F1=0.158). Applied to the test set, this threshold
produces F1=0.000, the model never crosses 0.40 on the 8 positive test
cases. Its ROC-AUC of 0.710 confirms it *ranks* delayed flights correctly,
but the threshold does not generalise. This is a direct consequence of the
synthetic, largely random data: there is no stable signal for the ensemble
to exploit across splits.

**The Business Trade-off (Recall vs. Precision):** The winning model
achieves **75% Recall**, catching 3 out of 4 actual delays. The trade-off
is low Precision (11.5%), meaning false alarms are frequent. In aviation,
missing a cascading delay is vastly more expensive than proactively alerting
a ground crew, making this high-recall approach operationally sound.

**Logistic Regression's stability** across val (F1=0.16) and test (F1=0.20)
is the most meaningful result here: it found a small but genuine and
generalisable signal, while the more complex models overfit to noise.

## 9. Business Interpretation

### What causes delays most?
Based on feature importances from Random Forest and XGBoost, the strongest signals are:
- `previous_leg_delay` — cascade effect: a delayed incoming flight is the strongest predictor
- `airport_congestion_index` — busy airports propagate delays
- `turnaround_minutes` — shorter turnarounds leave no buffer for recovery
- `crew_rest_hours` — fatigued crew correlates with operational issues
- Weather variables (`visibility`, `wind_speed`, `precipitation`) — limited controllability

### What can operations control?
| Controllable | Not controllable |
|---|---|
| Turnaround scheduling | Weather |
| Crew rest management | Air traffic control |
| Gate allocation | Route distance |
| Maintenance scheduling | Airport congestion (partially) |

### How much would I trust this model?
**Honestly not much, but for the right reasons.**

The data is synthetically generated with intentional randomness. A ROC-AUC of 0.75 from Logistic Regression on random data is actually a sign the model is finding the small amount of real signal that exists. On real operational data with genuine causal structure, the same pipeline would likely perform significantly better.

### Limitations
1. **Synthetic data** — no real causal relationships; results are not generalizable
2. **Tiny test set** — only 192 flights in May, of which ~8 are delayed; high variance
3. **Class imbalance** — 96/4 split requires careful threshold tuning in production
4. **No external data** — airline networks, NOTAM, slot constraints not included
5. **Static model** — flight patterns shift seasonally; model needs retraining schedule

### How to use in production?
- Run the model **T-60 minutes** before scheduled departure (all features available)
- Use a **lower decision threshold** (~0.15–0.20) to prioritise recall over precision — missing a real delay is more costly than a false alarm
- Monitor model **data drift** monthly; retrain quarterly on rolling 12-month window
- Use predictions as an **operational signal**, not a hard decision — route to dispatcher for confirmation


## 10. Conclusion

On this dataset, the most impactful decisions were made before any model
was trained: removing 5 leakage columns, recognising that accuracy is
meaningless at a 96/4 class split, applying a time-based split to reflect
real deployment conditions, and sweeping decision thresholds to reveal the
true predictive power hidden behind a conservative default of 0.5.

The resulting pipeline is honest about its limitations: the data is
synthetically generated, the test set contains only ~8 positive cases, and
the best F1 of 0.23 reflects that reality rather than a modelling failure.
The same pipeline applied to real operational flight data — with genuine
causal structure, historical delay chains, and external signals like NOTAM
or slot restrictions — would be expected to perform substantially better.

What this exercise does demonstrate is a production-ready analytical
approach: leakage-free, time-aware, imbalance-conscious, and grounded in
metrics that reflect actual business cost.